# Restart an Interrupted Extraction

A batch extraction over a large corpus can run for hours. If the process stops part way, the Amazon Bedrock batch jobs it submitted keep running, and their output lands in Amazon S3. Passing a `run_id` makes a run restartable: run it again with the same `run_id` and it takes up where the earlier run stopped.

This notebook starts a batch extraction, stops it while its Bedrock jobs are queued, shows what the run left in S3, and restarts it. It then shows a restart that is refused, builds a graph from the staged documents, and runs a query.

See [Restarting an interrupted extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/indexing/#restarting-an-interrupted-extraction). Restart covers batch extraction with a fixed batch size. A run with a `run_id` and `auto_tune=True` raises a `ValueError`.

## Setup

If you haven't already, install the toolkit and dependencies using the [Setup](./00-Setup.ipynb) notebook.

This notebook uses [batch extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/batch-extraction/), which the lexical-graph CloudFormation templates do not set up on their own. Deploy [`graphrag-toolkit-bedrock-batch.json`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/cloudformation-templates/graphrag-toolkit-bedrock-batch.json) in the same Region first. It creates a bucket for batch input and output, a role Bedrock assumes to run the jobs, and a policy that lets the notebook role submit, check and stop those jobs.

Then create or update the lexical-graph stack with the `BatchClientPolicyArn` output in its `IamPolicyArn` parameter, and add the other two outputs to your `.env` file:

```
export EXTRACTION_BUCKET=<BatchBucket output>
export BATCH_INFERENCE_ROLE=<BatchInferenceRole output>
```

The extraction model, `EXTRACTION_MODEL`, must support [Bedrock batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference-supported.html).


### Dataset

Bedrock batch inference needs at least 100 records in a job. With fewer, the toolkit extracts chunk by chunk instead, and there is no batch job to restart. This notebook uses the 100 aviation accident reports in [`examples/lexical-graph/workshop/assets/source-data/ntsb-md`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/workshop/assets/source-data/ntsb-md), taken from the [NTSB aviation investigation reports](https://www.ntsb.gov/investigations/AccidentReports/Pages/aviation.aspx). With the default chunking they split into about 1,100 chunks, so each of the two workers below submits one job of more than 500 records per extraction stage.

The next cell downloads the reports into `./ntsb-md`.

In [1]:
%%bash
if [ ! -d ntsb-md ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/awslabs/graphrag-toolkit.git graphrag-toolkit-src
  git -C graphrag-toolkit-src sparse-checkout set examples/lexical-graph/workshop/assets/source-data/ntsb-md
  cp -r graphrag-toolkit-src/examples/lexical-graph/workshop/assets/source-data/ntsb-md ./ntsb-md
fi
ls ntsb-md | wc -l

Cloning into 'graphrag-toolkit-src'...


100


## The extraction

The run is stopped by killing its process, so the first run has to run in a process of its own. The next cell writes the extraction to a file that both runs use. The first run and the restart then have the same settings, and the restart is not refused for a setting that differs by accident.

Each report is read as one document whose only metadata is its file name. A document's id is derived from its text and metadata, and a restart is refused when the ids differ from the plan, so the metadata must not change between runs. File dates, for example, would change if the reports were downloaded again.

`run_plan_store.staging_handler(run_id)` returns an `S3BasedDocs` handler for the collection that skips every source document already stored whole. On a first run it skips nothing.

In [2]:
%%writefile restart_extraction.py
import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, GraphRAGConfig, IndexingConfig, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.extract import BatchConfig, RunPlanStore

from llama_index.core.schema import Document

DOCS_DIR = 'ntsb-md'
KEY_PREFIX = 'restart-example'
EXTRACTION_BATCH_SIZE = 100


def run_plan_store(collection_id):
    return RunPlanStore(
        bucket_name=os.environ['EXTRACTION_BUCKET'],
        key_prefix=f'{KEY_PREFIX}/extracted',
        collection_id=collection_id
    )


def read_report(file_name):
    with open(os.path.join(DOCS_DIR, file_name)) as f:
        return Document(text=f.read(), metadata={'file_name': file_name})


def load_docs():
    return [read_report(name) for name in sorted(os.listdir(DOCS_DIR)) if name.endswith('.md')]


def extract(collection_id, run_id, batch_size=EXTRACTION_BATCH_SIZE):

    set_logging_config('INFO')

    GraphRAGConfig.extraction_batch_size = batch_size
    GraphRAGConfig.extraction_num_workers = 2

    store = run_plan_store(collection_id)

    indexing_config = IndexingConfig(
        batch_config=BatchConfig(
            region=GraphRAGConfig.aws_region,
            bucket_name=os.environ['EXTRACTION_BUCKET'],
            key_prefix=f'{KEY_PREFIX}/batch-inference',
            role_arn=os.environ['BATCH_INFERENCE_ROLE'],
            max_batch_size=1000,
            max_num_concurrent_batches=1
        )
    )

    with (
        GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
        VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
    ):

        graph_index = LexicalGraphIndex(
            graph_store,
            vector_store,
            indexing_config=indexing_config
        )

        graph_index.extract(
            load_docs(),
            handler=store.staging_handler(run_id),
            show_progress=True,
            run_id=run_id,
            run_plan_store=store
        )

Writing restart_extraction.py


The next cell picks a collection id and a run id for this notebook. Keep the run id: a restart is a run with the same `run_id`.

In [3]:
%reload_ext dotenv
%dotenv

import os
import uuid

from graphrag_toolkit.lexical_graph import GraphRAGConfig, set_logging_config
from graphrag_toolkit.lexical_graph.indexing.extract.batch_extractor_base import LIVE_JOB_STATES

from restart_extraction import KEY_PREFIX, extract, run_plan_store

set_logging_config('INFO')

bucket_name = os.environ['EXTRACTION_BUCKET']
collection_id = f'ntsb-{uuid.uuid4().hex[:8]}'
run_id = f'run-{uuid.uuid4().hex[:8]}'

store = run_plan_store(collection_id)
manifest_store = store.manifest_store(run_id)


def jobs_recorded():
    records = manifest_store.read_partitions(GraphRAGConfig.s3)
    return {partition: record for partition, record in records.items() if record.job_arn}


def job_status(record):
    return GraphRAGConfig.bedrock.get_model_invocation_job(jobIdentifier=record.job_arn)['status']


print(f'collection_id: {collection_id}')
print(f'run_id       : {run_id}')

collection_id: ntsb-9a4451f3
run_id       : run-4c4d844c


## Start a run and stop it

The next cell starts the first run in a child process with a session of its own, so that killing the session also kills the extraction workers. It waits until a partition record names a submitted Bedrock job, gives the other worker a minute to submit its job, and then kills the process. The child's output goes to `first-run.log`.

Killing the process does not stop the Bedrock jobs. They stay queued or running in Bedrock with nothing waiting on them.

In [4]:
import signal
import subprocess
import sys
import time

with open('first-run.log', 'w') as log:
    first_run = subprocess.Popen(
        [
            sys.executable, '-c',
            'import sys; from restart_extraction import extract; extract(sys.argv[1], sys.argv[2])',
            collection_id, run_id
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

try:
    while not jobs_recorded():
        if first_run.poll() is not None:
            raise RuntimeError(f'The first run ended before it submitted a job. See first-run.log [exit code: {first_run.returncode}]')
        time.sleep(10)
    time.sleep(60)
finally:
    if first_run.poll() is None:
        os.killpg(first_run.pid, signal.SIGKILL)
    first_run.wait()

print(f'First run stopped [exit code: {first_run.returncode}]')

First run stopped [exit code: -9]


## What the stopped run left in S3

The run keeps its records under `s3://<bucket>/restart-example/extracted/<collection_id>/_runs/<run_id>/`:

- `plan.json`, written before any job was submitted: the document ids in order, the batch size and the worker count. A restart divides the documents the same way, so it produces the same partitions.
- `partitions/<partition_id>.json`, one per partition, naming the Bedrock job it submitted and the attempt number.

Staged documents sit beside the run directory, one prefix per source document, each with a completion marker under `_markers/`. Documents are staged after both extraction stages finish, so a run stopped during its first batch jobs has staged nothing yet and there are no markers.

`plan_resume` reads these records and the collection and reports what a restart will reuse. It submits nothing.

In [5]:
from graphrag_toolkit.lexical_graph.indexing.extract.resume import plan_resume
from graphrag_toolkit.lexical_graph.indexing.load.s3_based_docs import is_completion_marker

plan = store.read(run_id, GraphRAGConfig.s3)
print(f'Plan: {len(plan.document_ids)} documents, batch_size: {plan.batch_size}, num_workers: {plan.num_workers}')
print()

interrupted = jobs_recorded()
for partition, record in interrupted.items():
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  job: {record.job_name}  Bedrock status: {job_status(record)}')
print()

pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(
    Bucket=bucket_name,
    Prefix=f'{store.key_prefix}/{collection_id}/'
)
keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
for key in keys:
    print(key)
print(f'Completion markers: {len([key for key in keys if is_completion_marker(key)])}')
print()

report = plan_resume(manifest_store, GraphRAGConfig.s3)
print(report.describe())

Plan: 100 documents, batch_size: 100, num_workers: 2

Partition 16e348bd82  attempt: 1  job: extract-propositions-run-4c4d844c-16e348bd82-a1-0-5a500  Bedrock status: Submitted
Partition 492b559452  attempt: 1  job: extract-propositions-run-4c4d844c-492b559452-a1-0-08939  Bedrock status: Submitted

restart-example/extracted/ntsb-9a4451f3/_runs/run-4c4d844c/partitions/16e348bd82e086fa699cd463a50645ce.json
restart-example/extracted/ntsb-9a4451f3/_runs/run-4c4d844c/partitions/492b559452e7d6a206c48ce7481c8c96.json
restart-example/extracted/ntsb-9a4451f3/_runs/run-4c4d844c/plan.json
restart-example/extracted/ntsb-9a4451f3/_staging.json
Completion markers: 0

2026-10-14 19:57:48:INFO:g.l.i.e.resume :Resuming a run [run_id: run-4c4d844c, partitions already done: 0, partitions to finish: 2, sources already staged in the collection: 0] Unfinished jobs: ['extract-propositions-run-4c4d844c-16e348bd82-a1-0-5a500', 'extract-propositions-run-4c4d844c-492b559452-a1-0-08939']
Resuming a run [run_id: ru

## Restart with the same run id

The restart reads the plan and the partition records. For each partition with a recorded job, it asks Bedrock for the job's status:

- A job still queued or running is waited on, rather than submitted again.
- A completed job has its output downloaded from S3.
- A job that failed, stopped or expired is submitted again as the next attempt.

Partitions the first run never reached, such as those of the second extraction stage, are submitted as usual. The cell runs until every job has finished, which depends on how long Bedrock takes to schedule and run them.

Afterwards, a partition whose recorded job completed still names that job and is still on attempt 1. When the run finishes, its partition records are folded into `rollup.json` in the run directory.

In [6]:
extract(collection_id, run_id)

for partition, record in manifest_store.read_partitions(GraphRAGConfig.s3).items():
    before = interrupted.get(partition)
    if before is None:
        submitted_by = 'submitted by the restart'
    elif before.job_arn == record.job_arn:
        submitted_by = 'same job as the first run'
    else:
        submitted_by = 'submitted again by the restart'
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  state: {record.state}  job: {record.job_name}  ({submitted_by})')

2026-10-14 19:57:49:INFO:g.l.i.e.resume :Resuming a run [run_id: run-4c4d844c, partitions already done: 0, partitions to finish: 2, sources already staged in the collection: 0] Unfinished jobs: ['extract-propositions-run-4c4d844c-16e348bd82-a1-0-5a500', 'extract-propositions-run-4c4d844c-492b559452-a1-0-08939']
2026-10-14 19:57:49:INFO:g.l.i.l.s3_based_docs:Not recording a collection that already holds objects, so it is read without completeness checking [collection_path: restart-example/extracted/ntsb-9a4451f3/]
2026-10-14 19:57:49:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch: 1/1, batch_size: 100, num_workers: 2]
2026-10-14 20:17:01:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 1, batch_size: 4, num_workers: 1, job_sizes: [37], batch_writes_enabled: True, batch_write_size: 25]
2026-10-14 20:17:02:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 2, batch_size: 4, num_workers: 1, job_sizes: [42], batch_writes_enabled: True, batch_write_size

## A changed setting is a new run

A restart has to divide the documents as the plan recorded. If `extraction_batch_size` or the set of documents differs, the restart is refused with a `RunPlanMismatch` before any work is submitted. To run with different settings, use a new `run_id`.

In [7]:
from graphrag_toolkit.lexical_graph.indexing.extract import RunPlanMismatch

try:
    extract(collection_id, run_id, batch_size=50)
except RunPlanMismatch as e:
    print(f'Refused: {e}')

2026-10-14 20:17:19:INFO:g.l.i.e.resume :Resuming a run [run_id: run-4c4d844c, partitions already done: 4, partitions to finish: 0, sources already staged in the collection: 100]
2026-10-14 20:17:19:INFO:g.l.i.l.s3_based_docs:Not recording a collection that already holds objects, so it is read without completeness checking [collection_path: restart-example/extracted/ntsb-9a4451f3/]
Refused: batch_size differs from the plan this run started with [recorded: 100, now: 50]. A changed batch_size is a new run, not a restart of this one.


## Build and query

The build stage reads the staged collection like any other `S3BasedDocs` collection, and skips the run directory. The graph is built under the `restart` tenant so that it stays apart from the graphs the other notebooks build. See [Multi-Tenancy](./04-Multi-Tenancy.ipynb).

In [8]:
from graphrag_toolkit.lexical_graph import LexicalGraphIndex, LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import S3BasedDocs

docs = S3BasedDocs(
    region=GraphRAGConfig.aws_region,
    bucket_name=bucket_name,
    key_prefix=store.key_prefix,
    collection_id=collection_id
)

with (
    GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
    VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
):

    graph_index = LexicalGraphIndex(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    graph_index.build(docs, show_progress=True)

    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    response = query_engine.query('What happened to the Bell 206 helicopter near Blair, Nebraska?')

print(response.response)

2026-10-14 20:17:20:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 1, batch_size: 4, num_workers: 2, job_sizes: [808, 853], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 808/808 [00:00<00:00, 17605.55it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 853/853 [00:00<00:00, 15937.20it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 808/808 [00:00<00:00, 109329.56it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 853/853 [00:00<00:00, 137078.21it/s]


2026-10-14 20:18:20:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 2, batch_size: 4, num_workers: 2, job_sizes: [922, 905], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 922/922 [00:00<00:00, 17249.42it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 905/905 [00:00<00:00, 16835.18it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 922/922 [00:00<00:00, 134870.72it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 905/905 [00:00<00:00, 134886.65it/s]


2026-10-14 20:18:47:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 3, batch_size: 4, num_workers: 2, job_sizes: [817, 778], batch_writes_enabled: True, batch_write_size: 25]


...

2026-10-14 20:28:16:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 24, batch_size: 4, num_workers: 2, job_sizes: [990, 850], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 990/990 [00:00<00:00, 17439.05it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 850/850 [00:00<00:00, 12580.17it/s]
[967d8/*] Retrying query in 4.307460243674675 seconds because it raised ConcurrentModificationException: An error occurred (ConcurrentModificationException) when calling the ExecuteOpenCypherQuery operation (reached max retries: 0): Operation failed due to conflicting concurrent operations (please retry), 0 transactions are currently rolling back. [attempt: 1, query: // insert facts, parameters:  -- 25 parameter set(s) -- ]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 850/850 [00:00<00:00, 130262.65it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 990/990 [00:00<00:00, 138448.95it/s]


2026-10-14 20:28:43:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch: 25, batch_size: 4, num_workers: 2, job_sizes: [1774, 816], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 1774/1774 [00:00<00:00, 16871.48it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 816/816 [00:00<00:00, 9649.64it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 816/816 [00:00<00:00, 121990.02it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 1774/1774 [00:00<00:00, 142793.72it/s]


Based on the search results, here is what is known about the Bell 206 helicopter incident near Blair, Nebraska:

- The aircraft involved was a **Bell 206B** helicopter (model/series 206 B), registered as **N106PD**, manufactured in 2008 (serial number 4649), powered by a **Rolls-Royce** turboshaft engine. [Source: 100068.md]

- The incident occurred at **Blair Municipal Airport (BTA)** in Blair, Nebraska, which has an elevation of 1,317 ft MSL. The flight departed at 0830 local time with no flight plan filed. [Source: 100068.md]

- The accident took place during **emergency training**, specifically what appears to have been an **autorotation practice**. The pilot-in-command was seated in the right seat, while the co-pilot (who was not a flight instructor) was seated in the left seat. [Source: 100068.md]

- During the training exercise, the **pilot-in-command rolled the throttle down just enough to take the governor offline**, and the helicopter descended while maintaining airspeed unti

## Clean up

The next cell stops any Bedrock job from this run that is still queued or running, then deletes the staged collection, the run records and the batch inference files under `restart-example/` in the bucket. It does not remove the `restart` tenant's data from the graph and vector stores, or the local files this notebook created: `ntsb-md`, `graphrag-toolkit-src`, `restart_extraction.py` and `first-run.log`.

In [9]:
for record in manifest_store.read_partitions(GraphRAGConfig.s3).values():
    if record.job_arn and job_status(record) in LIVE_JOB_STATES:
        GraphRAGConfig.bedrock.stop_model_invocation_job(jobIdentifier=record.job_arn)
        print(f'Stopped {record.job_name}')

for prefix in [f'{store.key_prefix}/{collection_id}/', f'{KEY_PREFIX}/batch-inference/']:
    pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(Bucket=bucket_name, Prefix=prefix)
    keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
    for i in range(0, len(keys), 1000):
        GraphRAGConfig.s3.delete_objects(
            Bucket=bucket_name,
            Delete={'Objects': [{'Key': key} for key in keys[i:i + 1000]]}
        )
    print(f'Deleted {len(keys)} objects under s3://{bucket_name}/{prefix}')

Deleted 1202 objects under s3://gr-nb-batch-batchbucket-i1yizjz7wwja/restart-example/extracted/ntsb-9a4451f3/
Deleted 16 objects under s3://gr-nb-batch-batchbucket-i1yizjz7wwja/restart-example/batch-inference/
